# OOP Capstone Project — Credit Risk Management System
### Build a complete, working system using every OOP concept from the last 4 notebooks

**This is the project that ties it all together.** You're going to build a small but genuinely realistic system for managing credit accounts and scoring risk — the same shape (if smaller) as real fintech backend code.

**What you'll use, and where:**
- **Classes & objects** — everything is built from classes
- **Encapsulation** — private balances, validated properties, static utility methods
- **Inheritance & polymorphism** — three account types sharing a base, each with their own fee/interest logic
- **Abstraction** — an enforced contract for account types AND for risk-scoring models

**How this notebook works:**
1. Each STAGE describes what to build and why.
2. You write the code yourself in the empty cell.
3. Run it, test it against the checks provided.
4. The FULL reference solution for the entire system is at the very end — don't peek until you've attempted every stage.

Take your time. This is meant to be the notebook that makes OOP click permanently, not another speed-drill.

---

## Stage 1 — The Abstract Base: `Account`

Design an abstract base class `Account(ABC)` with:
- `__init__(self, owner, balance=0)` storing `owner` publicly and `balance` as a PRIVATE attribute `self.__balance` (name-mangled)
- A `balance` `@property` (getter only — no setter, so external code can never directly overwrite it)
- An abstract method `monthly_fee(self)` — every account type must define its own fee
- An abstract method `account_type(self)` — every account type must name itself
- A CONCRETE method `deposit(self, amount)` that raises `ValueError` if `amount <= 0`, otherwise adds to `self.__balance`
- A CONCRETE method `withdraw(self, amount)` that raises `ValueError` if `amount <= 0` OR `amount > self.__balance`, otherwise subtracts
- A CONCRETE method `__str__(self)` returning `f"{self.account_type()} — {self.owner}: ${self.balance:,.2f}"` (notice it calls the ABSTRACT `account_type()` — this only works because abstraction GUARANTEES every subclass provides it)

**Why private balance + read-only property?** So the ONLY way balance can ever change is through `deposit`/`withdraw`, which validate. No code anywhere can do `account.balance = -9999` by accident.

In [ ]:
from abc import ABC, abstractmethod

# YOUR CODE HERE — build the Account class


**Self-check for Stage 1:** you should NOT be able to instantiate `Account()` directly (it's abstract). Try it below — this should raise `TypeError`.

In [ ]:
# try:
#     a = Account("Test", 100)
# except TypeError as e:
#     print("Correctly blocked:", e)


---
## Stage 2 — Three Concrete Account Types

Build three subclasses of `Account`, each implementing `monthly_fee()` and `account_type()`:

**`SavingsAccount(Account)`**
- `__init__(self, owner, balance, interest_rate)` — call `super().__init__(owner, balance)` first, then store `interest_rate`
- `monthly_fee()` returns `0` (no fee)
- `account_type()` returns `"Savings Account"`
- ADD a new method `apply_interest(self)` that calls `self.deposit(self.balance * self.interest_rate)` (reuses the validated deposit method rather than touching balance directly — this is encapsulation paying off)

**`CheckingAccount(Account)`**
- `__init__(self, owner, balance, overdraft_limit=0)`
- `monthly_fee()` returns `5` if `self.balance < 500` else `0` (fee waived above a minimum balance — logic depends on state)
- `account_type()` returns `"Checking Account"`

**`CreditCardAccount(Account)`**
- `__init__(self, owner, balance, credit_limit)` — note: for a credit card, `balance` represents amount OWED, and `deposit`/`withdraw` semantics get reused as "payment"/"charge" via two NEW methods:
  - `charge(self, amount)`: raises `ValueError` if `self.balance + amount > self.credit_limit`, otherwise increases balance (a charge adds to what's owed)
  - `pay(self, amount)`: reduces balance, not below 0
- `monthly_fee()` returns `15` if `self.balance / self.credit_limit > 0.9` (over 90% utilization) else `0`
- `account_type()` returns `"Credit Card"`
- ADD a `@staticmethod` `calculate_utilization(balance, limit)` returning `balance / limit`, and use it inside `monthly_fee()` instead of repeating the division inline

In [ ]:
# YOUR CODE HERE — build SavingsAccount, CheckingAccount, CreditCardAccount


**Self-check for Stage 2:**

In [ ]:
# s = SavingsAccount("Alice", 5000, 0.03)
# c = CheckingAccount("Bob", 300)
# cc = CreditCardAccount("Cara", 4500, 5000)
# print(s)
# print(c, c.monthly_fee())
# print(cc, cc.monthly_fee())


---
## Stage 3 — Polymorphism: A Portfolio Class

Build a `Portfolio` class (NOT related to `Account` by inheritance — it HOLDS accounts, it isn't one) with:
- `__init__(self)` creating `self.accounts = []`
- `add_account(self, account)` appending to the list (optionally: raise `TypeError` if the argument isn't an `Account` instance, using `isinstance`)
- `total_balance(self)` returning the sum of `.balance` across all held accounts
- `total_fees(self)` returning the sum of `.monthly_fee()` across all held accounts — THIS is pure polymorphism: `Portfolio` never needs to know or care which specific account types it's holding
- `accounts_by_type(self)` returning a dict `{account_type: count}` built by looping and calling `.account_type()` on each
- `__len__(self)` returning the number of accounts held (dunder method, so `len(portfolio)` works)
- `__str__(self)` returning a multi-line summary using the methods above

In [ ]:
# YOUR CODE HERE — build the Portfolio class


**Self-check for Stage 3:**

In [ ]:
# p = Portfolio()
# p.add_account(SavingsAccount("Alice", 5000, 0.03))
# p.add_account(CheckingAccount("Bob", 300))
# p.add_account(CreditCardAccount("Cara", 4500, 5000))
# print(len(p))
# print(p.total_balance())
# print(p.total_fees())
# print(p.accounts_by_type())
# print(p)


---
## Stage 4 — Abstraction: A Risk Scoring System

Separately from the account hierarchy, build a risk-scoring system:

- Abstract base class `RiskModel(ABC)` with:
  - Abstract method `score(self, account)` — takes an `Account` object (any subclass), returns a numeric risk score
  - Abstract method `model_name(self)`
  - CONCRETE method `evaluate(self, account)` returning a formatted string `f"[{self.model_name()}] {account.owner}: score={self.score(account):.1f}"`

- `BalanceRiskModel(RiskModel)`:
  - `score(self, account)` returns `max(0, 100 - account.balance / 100)` (lower balance = higher risk, simplified/synthetic formula — not real credit scoring math)
  - `model_name()` returns `"Balance Risk Model"`

- `UtilizationRiskModel(RiskModel)`:
  - `score(self, account)` — ONLY makes sense for `CreditCardAccount`. Use `isinstance(account, CreditCardAccount)` to check; if True, return `CreditCardAccount.calculate_utilization(account.balance, account.credit_limit) * 100`; if False, return `0` (not applicable, so no risk contribution from this model)
  - `model_name()` returns `"Utilization Risk Model"`

In [ ]:
# YOUR CODE HERE — build RiskModel, BalanceRiskModel, UtilizationRiskModel


**Self-check for Stage 4:**

In [ ]:
# balance_model = BalanceRiskModel()
# util_model = UtilizationRiskModel()
# cc = CreditCardAccount("Cara", 4500, 5000)
# print(balance_model.evaluate(cc))
# print(util_model.evaluate(cc))


---
## Stage 5 — Tying It Together: A Full Report Function

Write a standalone function `full_risk_report(portfolio, models)` that:
- Takes a `Portfolio` and a LIST of `RiskModel` instances
- For EVERY account in the portfolio, and EVERY model, prints `model.evaluate(account)`
- After all individual evaluations, prints the portfolio's `total_balance()`, `total_fees()`, and `accounts_by_type()`

This function should work correctly no matter how many account types or risk models exist — it never needs `isinstance` checks on account type, and only uses ONE `isinstance` check inside `UtilizationRiskModel` itself (where it's genuinely necessary, not scattered everywhere).

In [ ]:
# YOUR CODE HERE — build full_risk_report


**Self-check for Stage 5 — run your full system:**

In [ ]:
# portfolio = Portfolio()
# portfolio.add_account(SavingsAccount("Alice", 5000, 0.03))
# portfolio.add_account(CheckingAccount("Bob", 300))
# portfolio.add_account(CreditCardAccount("Cara", 4500, 5000))
#
# models = [BalanceRiskModel(), UtilizationRiskModel()]
# full_risk_report(portfolio, models)


---
## FULL REFERENCE SOLUTION
### Don't look until you've attempted all 5 stages yourself

In [ ]:
from abc import ABC, abstractmethod

class Account(ABC):
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.__balance = balance

    @property
    def balance(self):
        return self.__balance

    @abstractmethod
    def monthly_fee(self):
        pass

    @abstractmethod
    def account_type(self):
        pass

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit amount must be positive")
        self.__balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("Withdrawal amount must be positive")
        if amount > self.__balance:
            raise ValueError("Insufficient funds")
        self.__balance -= amount

    def __str__(self):
        return f"{self.account_type()} — {self.owner}: ${self.balance:,.2f}"


try:
    a = Account("Test", 100)
except TypeError as e:
    print("Correctly blocked:", e)

In [ ]:
class SavingsAccount(Account):
    def __init__(self, owner, balance, interest_rate):
        super().__init__(owner, balance)
        self.interest_rate = interest_rate

    def monthly_fee(self):
        return 0

    def account_type(self):
        return "Savings Account"

    def apply_interest(self):
        self.deposit(self.balance * self.interest_rate)


class CheckingAccount(Account):
    def __init__(self, owner, balance, overdraft_limit=0):
        super().__init__(owner, balance)
        self.overdraft_limit = overdraft_limit

    def monthly_fee(self):
        return 5 if self.balance < 500 else 0

    def account_type(self):
        return "Checking Account"


class CreditCardAccount(Account):
    def __init__(self, owner, balance, credit_limit):
        super().__init__(owner, balance)
        self.credit_limit = credit_limit

    @staticmethod
    def calculate_utilization(balance, limit):
        return balance / limit

    def charge(self, amount):
        if self.balance + amount > self.credit_limit:
            raise ValueError("Charge would exceed credit limit")
        self._Account__balance = self.balance + amount  # see note below

    def pay(self, amount):
        new_balance = max(self.balance - amount, 0)
        self._Account__balance = new_balance

    def monthly_fee(self):
        if CreditCardAccount.calculate_utilization(self.balance, self.credit_limit) > 0.9:
            return 15
        return 0

    def account_type(self):
        return "Credit Card"


s = SavingsAccount("Alice", 5000, 0.03)
c = CheckingAccount("Bob", 300)
cc = CreditCardAccount("Cara", 4500, 5000)
print(s)
print(c, c.monthly_fee())
print(cc, cc.monthly_fee())

**A note on the solution above:** `charge`/`pay` reach into `self._Account__balance` directly because the parent's name-mangled `__balance` isn't accessible as `self.__balance` from a subclass (mangling is per-class). In real code, the cleaner fix is to add PROTECTED (single-underscore) helper methods on `Account` itself — e.g. `_adjust_balance(self, delta)` — that subclasses can call instead of reaching past the mangling. That's a good improvement to try on your own once you've seen the working version.

In [ ]:
class Portfolio:
    def __init__(self):
        self.accounts = []

    def add_account(self, account):
        if not isinstance(account, Account):
            raise TypeError("Only Account instances can be added")
        self.accounts.append(account)

    def total_balance(self):
        return sum(a.balance for a in self.accounts)

    def total_fees(self):
        return sum(a.monthly_fee() for a in self.accounts)

    def accounts_by_type(self):
        counts = {}
        for a in self.accounts:
            t = a.account_type()
            counts[t] = counts.get(t, 0) + 1
        return counts

    def __len__(self):
        return len(self.accounts)

    def __str__(self):
        lines = [f"Portfolio: {len(self)} accounts"]
        lines.append(f"Total balance: ${self.total_balance():,.2f}")
        lines.append(f"Total monthly fees: ${self.total_fees():,.2f}")
        lines.append(f"By type: {self.accounts_by_type()}")
        return "\n".join(lines)


p = Portfolio()
p.add_account(SavingsAccount("Alice", 5000, 0.03))
p.add_account(CheckingAccount("Bob", 300))
p.add_account(CreditCardAccount("Cara", 4500, 5000))
print(len(p))
print(p.total_balance())
print(p.total_fees())
print(p.accounts_by_type())
print(p)

In [ ]:
class RiskModel(ABC):
    @abstractmethod
    def score(self, account):
        pass

    @abstractmethod
    def model_name(self):
        pass

    def evaluate(self, account):
        return f"[{self.model_name()}] {account.owner}: score={self.score(account):.1f}"


class BalanceRiskModel(RiskModel):
    def score(self, account):
        return max(0, 100 - account.balance / 100)

    def model_name(self):
        return "Balance Risk Model"


class UtilizationRiskModel(RiskModel):
    def score(self, account):
        if isinstance(account, CreditCardAccount):
            return CreditCardAccount.calculate_utilization(account.balance, account.credit_limit) * 100
        return 0

    def model_name(self):
        return "Utilization Risk Model"


balance_model = BalanceRiskModel()
util_model = UtilizationRiskModel()
cc = CreditCardAccount("Cara", 4500, 5000)
print(balance_model.evaluate(cc))
print(util_model.evaluate(cc))

In [ ]:
def full_risk_report(portfolio, models):
    for account in portfolio.accounts:
        for model in models:
            print(model.evaluate(account))
    print()
    print(f"Total balance: ${portfolio.total_balance():,.2f}")
    print(f"Total monthly fees: ${portfolio.total_fees():,.2f}")
    print(f"Accounts by type: {portfolio.accounts_by_type()}")


portfolio = Portfolio()
portfolio.add_account(SavingsAccount("Alice", 5000, 0.03))
portfolio.add_account(CheckingAccount("Bob", 300))
portfolio.add_account(CreditCardAccount("Cara", 4500, 5000))

models = [BalanceRiskModel(), UtilizationRiskModel()]
full_risk_report(portfolio, models)

---
## Final Reflection

Look back at what this one system used:

- **Encapsulation**: `Account.__balance` is private; the ONLY way to change it is through validated methods (`deposit`, `withdraw`), plus a read-only `balance` property and a `@staticmethod` utility (`calculate_utilization`)
- **Inheritance**: `SavingsAccount`, `CheckingAccount`, `CreditCardAccount` all share `Account`'s `deposit`/`withdraw`/`__str__` logic, adding only what's different
- **Polymorphism**: `Portfolio.total_fees()` and `full_risk_report()` never check WHICH account type they're looking at — they just call `.monthly_fee()` and trust every account type implements it correctly
- **Abstraction**: both `Account` and `RiskModel` use `ABC` + `@abstractmethod` to GUARANTEE every subclass implements the required contract, catching mistakes immediately instead of hoping someone remembers

**If you want to go further on your own** (recommended, not required): add a `LoanAccount` class, add a third `RiskModel` combining both existing scores, or fix the `charge`/`pay` name-mangling workaround mentioned above using a proper `_adjust_balance` protected method. Extending a working system yourself is the fastest way to make OOP permanent — much more than reading someone else's finished code.

**This closes out OOP.** You now have a real mental model for classes, encapsulation, inheritance, polymorphism, and abstraction — not just definitions, but a working system you built with them. Whenever you're ready, the next step is pandas.